### 1. Imports

In [1]:
import pandas as pd
from IPython.display import display, Markdown
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ParameterGrid

from pathlib import Path
src_dir = Path(r'D:\03_Python\sakura_git\src')
output_dir = Path('../../results/rf')

import sys
sys.path.insert(0, r'D:\03_Python\sakura_git\src')
sys.modules.pop('model_data', None)

from model_data import load_features, feature_columns, cv_folds, evaluate, TARGET


### 2. Create the four datasets

In [2]:
# import datasets
df_feb_large = load_features('feb1', source='csv')
df_mar_large = load_features('mar1', source='csv')
df_mar15_large = load_features('mar15', source='csv')

# filter datasets to only include stations in 2025
stations_2025 = df_mar_large.loc[df_mar_large['bloom_year'] == 2025,'station_id',].unique()
df_feb_small = df_feb_large[df_feb_large['station_id'].isin(stations_2025)].copy()
df_mar_small = df_mar_large[df_mar_large['station_id'].isin(stations_2025)].copy()
df_mar15_small = df_mar15_large[df_mar15_large['station_id'].isin(stations_2025)].copy()

# Execution order
datasets = {
    'Mar-15-large': df_mar15_large,
    'Mar-15-small': df_mar15_small,
    'Mar-large': df_mar_large,
    'Mar-small': df_mar_small,
    'Feb-large': df_feb_large,
    'Feb-small': df_feb_small,
}

# Display dataset summary
dataset_summary = []
for name, data in datasets.items():
    dataset_summary.append({
        'dataset': name,
        'rows': len(data),
        'stations': data['station_id'].nunique(),
        'development_rows': (data['split'] == 'development').sum(),
        'test_rows': (data['split'] == 'test').sum(),
        'features':len(feature_columns(data)),
    })
display(pd.DataFrame(dataset_summary))


Loaded model_features_feb1 from data/model/ CSV: 4,365 rows
Loaded model_features_mar1 from data/model/ CSV: 4,365 rows
Loaded model_features_mar15 from data/model/ CSV: 4,365 rows


,dataset,rows,stations,development_rows,test_rows,features
0,Mar-15-large,4365,82,3837,528,36
1,Mar-15-small,2972,48,2444,528,36
2,Mar-large,4365,82,3837,528,23
3,Mar-small,2972,48,2444,528,23
4,Feb-large,4365,82,3837,528,20
5,Feb-small,2972,48,2444,528,20


### 3. Set the shared parameter grid

In [14]:
param_grid = {
    'n_estimators': [200, 500],
    'max_depth': [5, 10, None],
    'min_samples_leaf': [1, 10, 30],
    'max_features': ['sqrt', 0.7, 1.0],
}

candidates = list(ParameterGrid(param_grid))

print('Candidates per dataset:', len(candidates))
print('Total model fits:', len(candidates) * 5 * len(datasets))

Candidates per dataset: 54
Total model fits: 1620


# 4. Grid search: predict deviations from the station's 30-year average

In [ ]:
# 4. Grid search: RF predicting deviations from a station reference
#
# Datasets: Mar15-large, Mar1-large, Feb1-large
# Training: ALL original training years in each fold
# Reference: latest 30 years of that fold's TRAINING data only
# Target: actual full-bloom DOY - station reference
# Evaluation: add reference back, then evaluate full-bloom DOY
# Hyperparameters: use the existing candidates
# Test data: not used

from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display, Markdown
from sklearn.ensemble import RandomForestRegressor

from model_data import feature_columns, cv_folds, evaluate, TARGET


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

# datasets = {
#     'Mar15-large': df_mar15_large,
#     'Mar1-large': df_mar_large,
#     'Feb1-large': df_feb_large,
# }

datasets = {
    'Mar15-small': df_mar15_small,
    'Mar1-small': df_mar_small,
    'Feb1-small': df_feb_small,
}

search_candidates = list(candidates)

if not search_candidates:
    raise ValueError('candidates is empty.')

STATION_REF_YEARS = 30
MIN_STATION_HISTORY = 10

# Use two CPU workers to leave capacity for other work.
N_JOBS = 2

output_dir = Path('../../results/rf')
output_dir.mkdir(parents=True, exist_ok=True)

anomaly_results = {}
comparison_rows = []
comparison_anomaly = pd.DataFrame()


# ------------------------------------------------------------
# Station reference: same calculation as the SVR implementation
# ------------------------------------------------------------

def make_station_reference(train):
    reference_end = int(train['bloom_year'].max())
    reference_start = reference_end - STATION_REF_YEARS + 1

    history = train.loc[
        train['bloom_year'].between(
            reference_start,
            reference_end,
        )
    ]

    station_stats = (
        history.groupby('station_id')[TARGET]
        .agg(['mean', 'size'])
    )

    region_mean = history.groupby('region')[TARGET].mean()
    overall_mean = history[TARGET].mean()

    def lookup(rows):
        station_mean = rows['station_id'].map(
            station_stats['mean']
        )

        station_count = (
            rows['station_id']
            .map(station_stats['size'])
            .fillna(0)
        )

        fallback = (
            rows['region']
            .map(region_mean)
            .fillna(overall_mean)
        )

        return np.where(
            station_count >= MIN_STATION_HISTORY,
            station_mean,
            fallback,
        ).astype(float)

    return lookup, reference_start, reference_end


print(
    f'{len(datasets)} datasets × '
    f'{len(search_candidates)} candidates × 5 folds = '
    f'{len(datasets) * len(search_candidates) * 5} fits',
    flush=True,
)

print(f'Save location: {output_dir.resolve()}', flush=True)


# ------------------------------------------------------------
# Run grid search separately for each dataset
# ------------------------------------------------------------

for dataset_name, df in datasets.items():
    features = feature_columns(df)

    folds = []
    period_rows = []

    # Prepare folds without shortening the training period.
    for fold, train, valid in cv_folds(df):
        assert not train.empty and not valid.empty
        assert (
            train['bloom_year'].max()
            < valid['bloom_year'].min()
        )
        assert valid['bloom_year'].max() <= 2014

        lookup, reference_start, reference_end = (
            make_station_reference(train)
        )

        train_reference = lookup(train)
        valid_reference = lookup(valid)

        y_train = train[TARGET].to_numpy(dtype=float)
        y_valid = valid[TARGET].to_numpy(dtype=float)

        folds.append({
            'fold': fold,
            'X_train': train[features],
            'X_valid': valid[features],
            'y_train': y_train,
            'y_valid': y_valid,
            'train_reference': train_reference,
            'valid_reference': valid_reference,
            'y_train_anomaly': y_train - train_reference,
        })

        period_rows.append({
            'fold': fold,
            'train_start': int(train['bloom_year'].min()),
            'train_end': int(train['bloom_year'].max()),
            'reference_start': reference_start,
            'reference_end': reference_end,
            'valid_start': int(valid['bloom_year'].min()),
            'valid_end': int(valid['bloom_year'].max()),
            'train_rows': len(train),
            'valid_rows': len(valid),
        })

    assert len(folds) == 5

    training_periods = pd.DataFrame(period_rows)

    display(Markdown(f'### {dataset_name}'))
    display(training_periods)

    print(f'Starting grid search: {dataset_name}', flush=True)

    fold_rows = []

    for candidate_id, params in enumerate(search_candidates):
        for data in folds:
            model = RandomForestRegressor(
                **params,
                random_state=42,
                n_jobs=N_JOBS,
            )

            # Learn the deviation from the fixed station reference.
            model.fit(
                data['X_train'],
                data['y_train_anomaly'],
            )

            # Convert predictions back to full-bloom DOY.
            train_prediction = (
                model.predict(data['X_train'])
                + data['train_reference']
            )

            valid_prediction = (
                model.predict(data['X_valid'])
                + data['valid_reference']
            )

            train_scores = evaluate(
                data['y_train'],
                train_prediction,
            )

            valid_scores = evaluate(
                data['y_valid'],
                valid_prediction,
            )

            fold_rows.append({
                'candidate_id': candidate_id,
                'fold': data['fold'],
                'train_MAE': train_scores['MAE'],
                'valid_MAE': valid_scores['MAE'],
                'valid_RMSE': valid_scores['RMSE'],
                'valid_R2': valid_scores['R2'],
                'valid_hit_rate_3d': valid_scores['hit_rate_3d'],
            })

    fold_results = pd.DataFrame(fold_rows)

    # Equal-weight mean across the five folds.
    mean_scores = (
        fold_results
        .groupby('candidate_id')
        .agg(
            train_MAE=('train_MAE', 'mean'),
            valid_MAE=('valid_MAE', 'mean'),
            valid_MAE_std=('valid_MAE', 'std'),
            valid_RMSE=('valid_RMSE', 'mean'),
            valid_R2=('valid_R2', 'mean'),
            valid_hit_rate_3d=('valid_hit_rate_3d', 'mean'),
        )
    )

    parameter_table = pd.DataFrame(search_candidates)
    parameter_table.index.name = 'candidate_id'

    gs_results = (
        parameter_table
        .join(mean_scores)
        .sort_values('valid_MAE', kind='stable')
    )

    best_id = int(gs_results.index[0])

    # Retrieve original values to preserve None/int parameter types.
    best_params = search_candidates[best_id].copy()

    best_fold_results = (
        fold_results.loc[
            fold_results['candidate_id'] == best_id
        ]
        .sort_values('fold')
        .reset_index(drop=True)
    )

    # Retain ALL candidate results in memory.
    anomaly_results[dataset_name] = {
        'features': features.copy(),
        'best_id': best_id,
        'best_params': best_params,
        'gs_results': gs_results,
        'fold_results': fold_results,
        'best_fold_results': best_fold_results,
        'training_periods': training_periods,
    }

    best = gs_results.loc[best_id]

    comparison_rows.append({
        'dataset': dataset_name,
        'target_mode': 'anomaly',
        'reference_years': STATION_REF_YEARS,
        'training_window': 'all',
        'features': len(features),
        'best_candidate_id': best_id,
        **best_params,
        'train_MAE': best['train_MAE'],
        'valid_MAE': best['valid_MAE'],
        'valid_MAE_std': best['valid_MAE_std'],
        'valid_RMSE': best['valid_RMSE'],
        'valid_R2': best['valid_R2'],
        'valid_hit_rate_3d': best['valid_hit_rate_3d'],
    })

    comparison_anomaly = (
        pd.DataFrame(comparison_rows)
        .set_index('dataset')
    )

    # --------------------------------------------------------
    # Save immediately after this dataset's search finishes.
    # --------------------------------------------------------

    file_prefix = (
        dataset_name.lower().replace('-', '_')
        + '_anomaly_ref30_trainall'
    )

    # 1. Every candidate's parameters and mean CV scores
    gs_results.to_csv(
        output_dir / f'{file_prefix}_grid_search_all_candidates_small.csv',
        index=True,
        index_label='candidate_id',
        encoding='utf-8-sig',
    )

    # 2. Every candidate's scores for every fold
    fold_results.to_csv(
        output_dir / f'{file_prefix}_all_candidates_fold_results_small.csv',
        index=False,
        encoding='utf-8-sig',
    )

    # 3. Best candidate's scores for every fold
    best_fold_results.to_csv(
        output_dir / f'{file_prefix}_best_candidate_fold_results_small.csv',
        index=False,
        encoding='utf-8-sig',
    )

    # 4. Best models across datasets: update after each completion
    comparison_anomaly.to_csv(
        output_dir / 'best_models_anomaly_ref30_trainall_comparison_small.csv',
        index=True,
        index_label='dataset',
        encoding='utf-8-sig',
    )

    print(
        f'Completed and saved: {dataset_name} '
        f'({len(anomaly_results)}/{len(datasets)})',
        flush=True,
    )

    print('Best parameters:', best_params)

    display(Markdown('#### Top 10 candidates'))
    display(gs_results.head(10))

    display(Markdown('#### Fold results for the best candidate'))
    display(best_fold_results)


# ------------------------------------------------------------
# 5. Display the final comparison of the three best models
# ------------------------------------------------------------

display(Markdown('### Best models: Mar15, Mar1, Feb1'))
display(comparison_anomaly.round(3))

3 datasets × 54 candidates × 5 folds = 810 fits
Save location: D:\03_Python\sakura_git\results\rf


### Mar15-large

,fold,train_start,train_end,reference_start,reference_end,valid_start,valid_end,train_rows,valid_rows
0,1,1964,2004,1975,2004,2005,2006,3262,148
1,2,1964,2006,1977,2006,2007,2008,3410,127
2,3,1964,2008,1979,2008,2009,2010,3537,106
3,4,1964,2010,1981,2010,2011,2012,3643,98
4,5,1964,2012,1983,2012,2013,2014,3741,96


Starting grid search: Mar15-large
Completed and saved: Mar15-large (1/3)
Best parameters: {'max_depth': 10, 'max_features': 0.7, 'min_samples_leaf': 1, 'n_estimators': 500}


#### Top 10 candidates

...

,max_depth,max_features,min_samples_leaf,n_estimators,train_MAE,valid_MAE,valid_MAE_std,valid_RMSE,valid_R2,valid_hit_rate_3d
candidate_id,,,,,,,,,,
4,5.0,sqrt,30,200,3.557159,3.784608,1.765547,4.551763,0.814686,0.474229
5,5.0,sqrt,30,500,3.557349,3.787267,1.757159,4.554717,0.814686,0.472099
1,5.0,sqrt,1,500,3.525972,3.789840,1.751452,4.556102,0.814961,0.470212
0,5.0,sqrt,1,200,3.527262,3.793031,1.761417,4.558747,0.814666,0.470678
3,5.0,sqrt,10,500,3.535560,3.802797,1.754288,4.567824,0.813955,0.476181
2,5.0,sqrt,10,200,3.535078,3.803331,1.750453,4.567787,0.814075,0.476404
6,5.0,0.7,1,200,3.478009,3.818909,1.787851,4.586251,0.811772,0.466532
11,5.0,0.7,30,500,3.500149,3.825784,1.782318,4.593650,0.811351,0.457552
16,5.0,1.0,30,200,3.490896,3.825963,1.776830,4.594246,0.811192,0.469039


#### Fold results for the best candidate

,candidate_id,fold,train_MAE,valid_MAE,valid_RMSE,valid_R2,valid_hit_rate_3d
0,4,1,3.591502,2.491129,3.334000,0.917883,0.716216
1,4,2,3.543500,3.776105,4.803307,0.732300,0.456693
2,4,3,3.553631,3.857245,4.752848,0.814391,0.462264
3,4,4,3.570471,2.159353,2.687654,0.924260,0.673469
4,4,5,3.526692,6.639207,7.181006,0.684596,0.062500


### Best models: Mar15, Mar1, Feb1

,target_mode,reference_years,training_window,features,best_candidate_id,max_depth,max_features,min_samples_leaf,n_estimators,train_MAE,valid_MAE,valid_MAE_std,valid_RMSE,valid_R2,valid_hit_rate_3d
dataset,,,,,,,,,,,,,,,
Mar15-large,anomaly,30,all,36,25,10,0.7,1,500,1.681,3.103,1.063,3.822,0.870,0.563
Mar1-large,anomaly,30,all,23,17,5,1.0,30,500,3.144,3.555,1.657,4.264,0.831,0.512
Feb1-large,anomaly,30,all,20,4,5,sqrt,30,200,3.557,3.785,1.766,4.552,0.815,0.474
